# Feature Track 3: Maintain a Conversation

In a multi-turn dialogue the assistant must remember what was said earlier. This notebook explores three mechanisms for passing conversation history to a RAG agent:

| # | Strategy | Idea |
|---|---|---|
| 1 | **Full history** | Append every exchange verbatim to the context window |
| 2 | **History summarization** | Compress prior turns into a short summary before each query |
| 3 | **Query reformulation** | Rewrite the user query to be self-contained before retrieval |

---

## Setup

**Prerequisites:** `conversational_toolkit` and `backend` must be installed in editable mode.
For **OpenAI** set `OPENAI_API_KEY`. For **Ollama** start `ollama serve` and pull the model.

In [ ]:
from loguru import logger
import sys

from conversational_toolkit.llms.base import LLMMessage, Roles, MessageContent
from conversational_toolkit.agents.base import QueryWithContext
from conversational_toolkit.agents.rag import RAG
from conversational_toolkit.embeddings.openai import OpenAIEmbeddings
from conversational_toolkit.vectorstores.chromadb import ChromaDBVectorStore

from sme_kt_zh_collaboration_rag.feature0_baseline_rag import (
    load_chunks,
    build_vector_store,
    build_llm,
    build_agent,
    SYSTEM_PROMPT,
    EMBEDDING_MODEL,
    VS_PATH,
)

logger.remove()
logger.add(sys.stderr, level="WARNING")  # hides DEBUG

# Choose your LLM backend (only needed for the final answer cells)
BACKEND = "openai"  # "ollama", "openai", or "qwen"
FORCE_REBUILD = False  # set True to re-embed from scratch and rebuild the vector store

embedding_model = OpenAIEmbeddings(model_name=EMBEDDING_MODEL)
print(f"Embedding model: {EMBEDDING_MODEL}")

if FORCE_REBUILD or not VS_PATH.exists():
    chunks = load_chunks(max_files=5)
    chunks = [c for c in chunks if c.mime_type.startswith("text")]
    vector_store = await build_vector_store(
        chunks, embedding_model, db_path=VS_PATH, reset=FORCE_REBUILD
    )
else:
    # Vector store already exists -> open it without re-embedding
    vector_store = ChromaDBVectorStore(db_path=str(VS_PATH))
    print(
        f"Reusing existing vector store at {VS_PATH} ({vector_store.collection.count()} chunks)"
    )

llm = build_llm(backend=BACKEND)
print(f"LLM backend: {BACKEND}")

In [ ]:
agent = build_agent(
    vector_store,
    embedding_model,
    llm,
    top_k=5,
    system_prompt=SYSTEM_PROMPT,
)

### Single-turn baseline

A single query with an empty history -> the starting point before any multi-turn logic is introduced.

In [ ]:
query = "Which pallets in our portfolio have a third-party verified EPD?"
query_with_context = QueryWithContext(query=query, history=[])

response = await agent.answer(query_with_context)

print("Answer:")
print(f"{response.content}")
print(f"Sources ({len(response.sources)}):")
for src in response.sources:
    source_file = src.metadata.get("source_file", "?")
    print(f"  {source_file!r}  |  {src.title!r}")

---

## Strategy 1: Full History

The simplest approach is to pass the entire conversation history to the agent on every turn. Each exchange (user query + assistant answer) is appended as a pair of `LLMMessage` objects and sent verbatim in the next request.

**Trade-off:** Works well for short conversations. As the history grows it consumes more of the context window, increases cost, and can degrade retrieval quality because the retriever must rank chunks against an increasingly long prompt.

> Note: retrieved source chunks are intentionally *not* included in the history. Storing full chunks would make the context balloon quickly.

Three queries are run in sequence. After each answer the user and assistant turns are appended to `history`, which is passed to the agent on the next call. The third query ("Make minutes of our conversation") only makes sense with history — it would fail without it.

In [ ]:
async def run_multi_turn_conversation(query_list: list[str], agent: RAG):
    history: list[LLMMessage] = []

    for query in query_list:
        response = await agent.answer(QueryWithContext(query=query, history=history))

        print("_" * 80)
        print("\nHistory:")
        for msg in history:
            print(f"{msg.role.value}: {msg.content[0].text}")
            print("-" * 40)
        print("\nAnswer:")
        print(f"{response.content[0].text}")
        print(f"\nSources ({len(response.sources)}):")
        for src in response.sources:
            source_file = src.metadata.get("source_file", "?")
            print(f"  {source_file!r} | {src.title!r}")

        response_content = response.content[0].text
        history.append(
            LLMMessage(
                content=[MessageContent(type="text", text=query)], role=Roles.USER
            )
        )
        history.append(
            LLMMessage(
                content=[MessageContent(type="text", text=response_content)],
                role=Roles.ASSISTANT,
            )
        )

In [ ]:
query_l = [
    "Which pallets in our portfolio have a third-party verified EPD?",
    "Rewrite what you just said as a pirate and very concisely.",
    "Make minutes of our conversation so far, you are 'pAIrate' and I'm AIvan",
]
await run_multi_turn_conversation(query_list=query_l, agent=agent)

---

## Strategy 2: History Summarization

Instead of sending the full history, an LLM (`gpt-4o-mini`) is called before each turn to compress prior exchanges into a single summary message. Only that summary, not the raw history, is injected into the next request as a `SYSTEM` message.

**Trade-off:** Keeps the context window small and cost low, but lossy -> fine-grained details from earlier turns may be dropped. Good when the conversation is long but only the gist of prior turns matters.

In [ ]:
async def summarize_history(history: list[LLMMessage]) -> list[MessageContent]:
    llm_summarization = build_llm("openai", model_name="gpt-4o-mini")

    summarization_system_prompt = "You are a helpful assistant that summarizes conversations between a user and an assistant. Do not answer any question, just summarize the conversation in a concise way. The user is called 'User' and the assistant is called 'Assistant'."
    messages: list[LLMMessage] = [
        LLMMessage(
            content=[MessageContent(type="text", text=summarization_system_prompt)],
            role=Roles.SYSTEM,
        ),
        *history,
    ]

    msg = await llm_summarization.generate(conversation=messages)
    return msg.content


async def run_multi_turn_conversation_history_summary(
    query_list: list[str], agent: RAG
):
    history: list[LLMMessage] = []
    summaries: list[list[MessageContent]] = []

    for query in query_list:
        print("_" * 80)
        if len(history):
            summary_msg = await summarize_history(history)
            print(f"\nSummary: {summary_msg[0].text}")
            summaries.append(summary_msg)
            history_tmp = [LLMMessage(content=summary_msg, role=Roles.SYSTEM)]
        else:
            history_tmp = []

        response = await agent.answer(
            QueryWithContext(query=query, history=history_tmp)
        )
        response_content = response.content[0].text

        print("\nAnswer:")
        print(f"{response_content}")
        print(f"\nSources ({len(response.sources)}):")
        for src in response.sources:
            source_file = src.metadata.get("source_file", "?")
            print(f"  {source_file!r}  |  {src.title!r}")

        history.append(
            LLMMessage(
                content=[MessageContent(type="text", text=query)], role=Roles.USER
            )
        )
        history.append(
            LLMMessage(
                content=[MessageContent(type="text", text=response_content)],
                role=Roles.ASSISTANT,
            )
        )

### Summarization loop

On the first turn no summary exists yet, so the history is empty. From the second turn onward `summarize_history` compresses all prior exchanges and the result is passed as the sole context message.

In [ ]:
query_l = [
    "Which pallets in our portfolio have a third-party verified EPD?",
    "Rewrite what you just said as a pirate and very concisely.",
    "What is the current customer request?",
]

await run_multi_turn_conversation_history_summary(query_list=query_l, agent=agent)

---

## Strategy 3: Query Reformulation

When a user says "What is the meaning of code?" in the context of an EPD conversation, the retriever has no way to know they mean "EPD code" without the history. Query reformulation solves this by asking an LLM to rewrite the query into a self-contained form before embedding it for retrieval.

This is already wired into the agent via `make_query_standalone` in `conversational_toolkit.utils.retriever`. Enabling `DEBUG` logging below makes the reformulation step visible.

In [ ]:
logger.add(sys.stderr, level="DEBUG")

query_l = [
    "Hi, I'm Ivan! What is the pallet which has a 3rd-party verified EPD?",
    "What is the meaning of code?",
]

history: list[LLMMessage] = []

for query in query_l:
    print("_" * 80)
    response = await agent.answer(QueryWithContext(query=query, history=history))
    response_content = response.content[0].text

    print("\nAnswer:")
    print(f"{response_content}")
    print(f"\nSources ({len(response.sources)}):")
    for src in response.sources:
        source_file = src.metadata.get("source_file", "?")
        print(f"  {source_file!r}  |  {src.title!r}")

    history.append(
        LLMMessage(content=[MessageContent(type="text", text=query)], role=Roles.USER)
    )
    history.append(
        LLMMessage(
            content=[MessageContent(type="text", text=response_content)],
            role=Roles.ASSISTANT,
        )
    )